In [ ]:
import pandas as pd



# Reload latest processed dataset

df = pd.read_csv("../data/processed_internships.csv")

print(df.columns.tolist())

In [2]:
# Step 2 - Internship Features

internship_features = [
    "internship_id",
    "role",
    "company",
    "location",
    "skills",
    "skills_list",
    "eligibility",
    "stipend_min",
    "stipend_max",
    "stipend_avg",
    "duration_months",
    "start_type",
    "deadline",
    "description",
    "domain",
    "work_mode",
    "internship_type"
]

internship_df = df[internship_features].copy()

print("Shape:", internship_df.shape)

print("\nColumns:")
print(internship_df.columns.tolist())

Shape: (200, 17)

Columns:
['internship_id', 'role', 'company', 'location', 'skills', 'skills_list', 'eligibility', 'stipend_min', 'stipend_max', 'stipend_avg', 'duration_months', 'start_type', 'deadline', 'description', 'domain', 'work_mode', 'internship_type']


In [3]:
#step 3
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pretrained semantic embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Semantic NLP model loaded successfully.")

c:\Users\LENOVO\Desktop\AI ML\sklearn-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1608.19it/s]


Semantic NLP model loaded successfully.


In [4]:
import ast

In [5]:
# Convert skills_list strings back into Python lists

internship_df["skills_list"] = internship_df["skills_list"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Check
print(internship_df["skills_list"].head(5))
print(type(internship_df["skills_list"].iloc[0]))

0           [html, css, javascript, react.js, node.js]
1    [figma, ui design, ux design, wireframing, pro...
2        [html, css, javascript, react.js, typescript]
3        [html, css, javascript, react.js, typescript]
4             [dart, flutter, firebase, rest api, git]
Name: skills_list, dtype: object
<class 'list'>


In [6]:
# Step 4 - Create semantic embeddings for internship skills

# Collect all unique skills from all internships
all_skills = set()

for skill_list in internship_df["skills_list"]:
    if isinstance(skill_list, list):
        all_skills.update(skill_list)

all_skills = sorted(all_skills)

print("Total unique internship skills:", len(all_skills))
print("\nFirst 20 skills:")
print(all_skills[:20])

Total unique internship skills: 162

First 20 skills:
['3d modeling', 'airflow', 'analytical thinking', 'android sdk', 'android studio', 'apache spark', 'api testing', 'arduino', 'artificial intelligence', 'attention to detail', 'autocad', 'automation testing', 'aws', 'azure', 'b2b sales', 'bash', 'bert', 'blogging', 'business management', 'business research']


In [7]:
# Generate embeddings for all unique internship skills

skill_embeddings = model.encode(
    all_skills,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Store skill -> embedding mapping
skill_embedding_map = {
    skill: embedding
    for skill, embedding in zip(all_skills, skill_embeddings)
}

print("Embeddings created:", len(skill_embedding_map))

# Check one example
sample_skill = all_skills[0]

print("\nSample skill:", sample_skill)
print("Embedding size:", len(skill_embedding_map[sample_skill]))

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches: 100%|██████████| 6/6 [00:01<00:00,  3.74it/s]

Embeddings created: 162

Sample skill: 3d modeling
Embedding size: 384


In [8]:
# Step 5 - Student skills

student_skills = [
    "python",
    "ml",
    "sql"
]

# Convert student skills into embeddings
student_skill_embeddings = model.encode(
    student_skills,
    normalize_embeddings=True
)

print("Student skills:", student_skills)
print("Number of embeddings:", len(student_skill_embeddings))
print("Embedding size:", student_skill_embeddings.shape[1])

Student skills: ['python', 'ml', 'sql']
Number of embeddings: 3
Embedding size: 384


In [9]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Example internship skills
internship_skills = ["python", "machine learning", "mysql"]

# Generate embeddings for internship skills
internship_skill_embeddings = model.encode(
    internship_skills,
    normalize_embeddings=True
)

# Compare every student skill with every internship skill
similarity_matrix = cosine_similarity(
    student_skill_embeddings,
    internship_skill_embeddings
)

print("Similarity Matrix:")
print(similarity_matrix)

Similarity Matrix:
[[0.9999999  0.3613274  0.3051863 ]
 [0.28822848 0.3726635  0.22862701]
 [0.29998446 0.31689715 0.68738586]]


In [10]:
best_matches = similarity_matrix.max(axis=1)

for skill, score in zip(student_skills, best_matches):
    print(f"{skill} → {score:.3f}")

python → 1.000
ml → 0.373
sql → 0.687


In [11]:
# Step 7 - Skill normalization / alias mapping

SKILL_ALIASES = {
    "ml": "machine learning",
    "ai/ml": "machine learning",
    "machine learning": "machine learning",

    "js": "javascript",
    "javascript": "javascript",

    "node": "node.js",
    "nodejs": "node.js",
    "node.js": "node.js",

    "reactjs": "react",
    "react.js": "react",
    "react": "react",

    "cpp": "c++",
    "c++": "c++"
}

def normalize_skill(skill):
    skill = str(skill).strip().lower()
    return SKILL_ALIASES.get(skill, skill)


# Test
test_skills = [
    "ML",
    "Machine Learning",
    "JS",
    "NodeJS",
    "React.js",
    "Python"
]

normalized = [normalize_skill(skill) for skill in test_skills]

for original, normalized_skill in zip(test_skills, normalized):
    print(f"{original:20} → {normalized_skill}")

ML                   → machine learning
Machine Learning     → machine learning
JS                   → javascript
NodeJS               → node.js
React.js             → react
Python               → python


In [12]:
# s-8 Apply skill normalization to all internship skills

internship_df["normalized_skills"] = internship_df["skills_list"].apply(
    lambda skills: [normalize_skill(skill) for skill in skills]
)

# Check original vs normalized skills
display(
    internship_df[["skills_list", "normalized_skills"]].head(10)
)

,skills_list,normalized_skills
0,"[html, css, javascript, react.js, node.js]","[html, css, javascript, react, node.js]"
1,"[figma, ui design, ux design, wireframing, pro...","[figma, ui design, ux design, wireframing, pro..."
2,"[html, css, javascript, react.js, typescript]","[html, css, javascript, react, typescript]"
3,"[html, css, javascript, react.js, typescript]","[html, css, javascript, react, typescript]"
4,"[dart, flutter, firebase, rest api, git]","[dart, flutter, firebase, rest api, git]"
5,"[python, node.js, express.js, rest api, sql]","[python, node.js, express.js, rest api, sql]"
6,"[python, sql, pandas, power bi, statistics]","[python, sql, pandas, power bi, statistics]"
7,"[html, css, javascript, react.js, typescript]","[html, css, javascript, react, typescript]"
8,"[python, pytorch, tensorflow, deep learning, nlp]","[python, pytorch, tensorflow, deep learning, nlp]"
9,"[kotlin, java, android studio, android sdk, fi...","[kotlin, java, android studio, android sdk, fi..."


In [13]:
# Student skills
student_skills = [
    "python",
    "ML",
    "SQL"
]

# Normalize student skills
normalized_student_skills = [
    normalize_skill(skill)
    for skill in student_skills
]

print("Original skills:")
print(student_skills)

print("\nNormalized skills:")
print(normalized_student_skills)

Original skills:
['python', 'ML', 'SQL']

Normalized skills:
['python', 'machine learning', 'sql']


In [14]:
# Step 10 - Semantic skill matching

def calculate_skill_match(student_skills, internship_skills, model):
    if not student_skills or not internship_skills:
        return 0.0

    # Create embeddings
    student_embeddings = model.encode(
        student_skills,
        normalize_embeddings=True
    )

    internship_embeddings = model.encode(
        internship_skills,
        normalize_embeddings=True
    )

    # Similarity matrix
    similarity_matrix = cosine_similarity(
        student_embeddings,
        internship_embeddings
    )

    # For every student skill, take its best internship-skill match
    best_matches = similarity_matrix.max(axis=1)

    # Average best-match similarity
    return float(np.mean(best_matches))


# Test on one example internship
sample_internship = internship_df.iloc[0]

score = calculate_skill_match(
    normalized_student_skills,
    sample_internship["normalized_skills"],
    model
)

print("Student Skills:", normalized_student_skills)
print("Internship Skills:", sample_internship["normalized_skills"])
print(f"Skill Match Score: {score:.3f}")

Student Skills: ['python', 'machine learning', 'sql']
Internship Skills: ['html', 'css', 'javascript', 'react', 'node.js']
Skill Match Score: 0.311


In [15]:
# Step 11 - Calculate Skill Match Score for All Internships

# Collect all unique normalized skills
all_normalized_skills = sorted({
    skill
    for skills in internship_df["normalized_skills"]
    if isinstance(skills, list)
    for skill in skills
})

print("Total unique normalized skills:", len(all_normalized_skills))


# Create embeddings for all normalized skills
normalized_skill_embeddings = model.encode(
    all_normalized_skills,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Create skill -> embedding mapping
normalized_skill_embedding_map = {
    skill: embedding
    for skill, embedding in zip(
        all_normalized_skills,
        normalized_skill_embeddings
    )
}


# Student skill embeddings
student_embeddings = model.encode(
    normalized_student_skills,
    normalize_embeddings=True
)


# Function to calculate skill similarity using precomputed embeddings
def calculate_skill_score(student_skills, internship_skills):
    
    if not student_skills or not internship_skills:
        return 0.0

    # Get internship embeddings
    internship_embeddings = np.array([
        normalized_skill_embedding_map[skill]
        for skill in internship_skills
        if skill in normalized_skill_embedding_map
    ])

    if len(internship_embeddings) == 0:
        return 0.0

    # Similarity matrix
    similarity_matrix = cosine_similarity(
        student_embeddings,
        internship_embeddings
    )

    # Best matching internship skill for each student skill
    best_matches = similarity_matrix.max(axis=1)

    # Average similarity
    return float(np.mean(best_matches))


# Calculate score for every internship
internship_df["skill_match_score"] = internship_df[
    "normalized_skills"
].apply(
    lambda skills: calculate_skill_score(
        normalized_student_skills,
        skills
    )
)


# Check results
print("\nSkill Match Scores:")
print(
    internship_df[
        ["internship_id", "role", "normalized_skills", "skill_match_score"]
    ].head(10)
)

Total unique normalized skills: 162


Batches: 100%|██████████| 6/6 [00:02<00:00,  2.36it/s]



Skill Match Scores:
  internship_id                     role  \
0  2025WSHP0080     Full Stack Developer   
1  2025WSHP0165             UI/UX Design   
2  2025WSHP0242       Frontend Developer   
3  2025WSHP0016       Frontend Developer   
4  2025WSHP0513      Flutter Development   
5  2025WSHP0071        Backend Developer   
6  2025WSHP0149           Data Analytics   
7  2025WSHP0696       Frontend Developer   
8  2025WSHP0827  Artificial Intelligence   
9  2025WSHP0182      Android Development   

                                   normalized_skills  skill_match_score  
0            [html, css, javascript, react, node.js]           0.311240  
1  [figma, ui design, ux design, wireframing, pro...           0.253464  
2         [html, css, javascript, react, typescript]           0.311240  
3         [html, css, javascript, react, typescript]           0.311240  
4           [dart, flutter, firebase, rest api, git]           0.254203  
5       [python, node.js, express.js, rest api, sq

In [16]:
# Step 12 - Analyze Skill Match Score Distribution

print("Skill Match Score Statistics:")
print(internship_df["skill_match_score"].describe())

print("\nLowest Skill Match Scores:")
print(
    internship_df[
        ["internship_id", "role", "skill_match_score"]
    ]
    .sort_values("skill_match_score")
    .head(10)
)

print("\nHighest Skill Match Scores:")
print(
    internship_df[
        ["internship_id", "role", "skill_match_score"]
    ]
    .sort_values("skill_match_score", ascending=False)
    .head(10)
)

Skill Match Score Statistics:
count    200.000000
mean       0.443394
std        0.178788
min        0.134600
25%        0.305935
50%        0.370076
75%        0.573861
max        0.801445
Name: skill_match_score, dtype: float64

Lowest Skill Match Scores:
    internship_id                              role  skill_match_score
186  2025WSHP0077                             sales           0.134600
181  2025WSHP0007                       Telecalling           0.203943
190  2025WSHP0076                  Event Management           0.203943
184  2025WSHP0147                         Law/Legal           0.211613
177  2025WSHP0146               Sales and Marketing           0.216827
182  2025WSHP0012                       Field Sales           0.216827
108   2026INT0109               Android Development           0.220002
106   2026INT0107                   Cloud Computing           0.238740
180  2025WSHP0048  Search Engine Optimization (SEO)           0.239381
178  2025WSHP0021               

In [17]:
# Step 13 - Convert semantic skill similarity to 0-100 scale

internship_df["skill_match_score_100"] = (
    internship_df["skill_match_score"] * 100
)

print(
    internship_df[
        ["role", "skill_match_score", "skill_match_score_100"]
    ].head(10)
)

                      role  skill_match_score  skill_match_score_100
0     Full Stack Developer           0.311240              31.123963
1             UI/UX Design           0.253464              25.346366
2       Frontend Developer           0.311240              31.123963
3       Frontend Developer           0.311240              31.123963
4      Flutter Development           0.254203              25.420347
5        Backend Developer           0.787109              78.710914
6           Data Analytics           0.801445              80.144525
7       Frontend Developer           0.311240              31.123963
8  Artificial Intelligence           0.662891              66.289055
9      Android Development           0.377261              37.726125


In [18]:
# Step 14 - Calculate Role Semantic Similarity

# Sample student desired role
student_role = "Machine Learning Engineer"

# Create embedding for student role
student_role_embedding = model.encode(
    [student_role],
    normalize_embeddings=True
)

# Create embeddings for all internship roles
internship_role_embeddings = model.encode(
    internship_df["role"].fillna("").tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# Calculate similarity between student role and every internship role
role_similarity = cosine_similarity(
    student_role_embedding,
    internship_role_embeddings
)[0]

# Store raw role similarity
internship_df["role_similarity"] = role_similarity

# Convert to 0-100 scale
internship_df["role_similarity_100"] = (
    internship_df["role_similarity"] * 100
)

# View results
print(
    internship_df[
        ["role", "role_similarity", "role_similarity_100"]
    ]
    .sort_values("role_similarity", ascending=False)
    .head(10)
)
internship_df.columns

Batches: 100%|██████████| 7/7 [00:01<00:00,  4.88it/s]


                        role  role_similarity  role_similarity_100
14          Machine Learning         0.617557            61.755680
158         Machine Learning         0.617557            61.755680
127         Machine Learning         0.617557            61.755680
96          Machine Learning         0.617557            61.755680
65          Machine Learning         0.617557            61.755676
68   Artificial Intelligence         0.534259            53.425938
8    Artificial Intelligence         0.534259            53.425938
37   Artificial Intelligence         0.534259            53.425938
130  Artificial Intelligence         0.534259            53.425938
161  Artificial Intelligence         0.534259            53.425938


Index(['internship_id', 'role', 'company', 'location', 'skills', 'skills_list',
       'eligibility', 'stipend_min', 'stipend_max', 'stipend_avg',
       'duration_months', 'start_type', 'deadline', 'description', 'domain',
       'work_mode', 'internship_type', 'normalized_skills',
       'skill_match_score', 'skill_match_score_100', 'role_similarity',
       'role_similarity_100'],
      dtype='str')

In [19]:
# Step 15 - Inspect internship roles for domain classification

print("Total unique roles:", internship_df["role"].nunique())

print("\nAll Internship Roles:")
for i, role in enumerate(internship_df["role"].dropna().unique(), start=1):
    print(f"{i:3}. {role}")

Total unique roles: 61

All Internship Roles:
  1. Full Stack Developer
  2. UI/UX Design
  3. Frontend Developer
  4. Flutter Development
  5. Backend Developer
  6. Data Analytics
  7. Artificial Intelligence
  8. Android Development
  9. Software Development
 10. Embedded Systems
 11. Cybersecurity
 12. Machine Learning
 13. Generative AI
 14. Software Testing
 15. Robotics
 16. Game Development
 17. Data Science
 18. React Native Development
 19. Database Engineering
 20. iOS Development
 21. CAD Design
 22. Data Engineering
 23. Cloud Computing
 24. Blockchain Development
 25. DevOps
 26. NLP
 27. Computer Vision
 28. Deep Learning
 29. IoT Development
 30. VLSI Design
 31. System Programming
 32. Business Development (Sales)
 33. Digital Marketing
 34. Human Resources (HR)
 35. Content and Social Media Marketing
 36. Social Media Marketing
 37. Video Editing/Making
 38. Marketing
 39. Sales and Marketing
 40. Content Writing
 41. Operations
 42. Search Engine Optimization (SEO)
 

In [20]:
# Step 15 - Create domain representations for semantic classification

domain_descriptions = {
    "Software Development":
        "Software engineering, application development, programming, backend development, "
        "mobile app development, software testing, system programming and building software applications.",

    "AI / Machine Learning":
        "Artificial intelligence, machine learning, deep learning, generative AI, neural networks, "
        "computer vision, natural language processing and intelligent systems.",

    "Data Science":
        "Data science, data analytics, data engineering, statistics, data processing, "
        "business intelligence, dashboards and extracting insights from data.",

    "Web Development":
        "Web development, frontend development, full stack development, websites, "
        "HTML, CSS, JavaScript, React, Node.js and web applications.",

    "Cyber Security":
        "Cybersecurity, information security, network security, ethical hacking, "
        "penetration testing, security monitoring and protecting systems.",

    "Cloud Computing":
        "Cloud computing, cloud infrastructure, AWS, Azure, Google Cloud, DevOps, "
        "cloud deployment, containers and scalable cloud services.",

    "UI / UX":
        "UI design, UX design, user experience, user interface, Figma, wireframing, "
        "prototyping, usability and visual interface design."
}

domains = list(domain_descriptions.keys())

print("Domains:")
for domain in domains:
    print("-", domain)

Domains:
- Software Development
- AI / Machine Learning
- Data Science
- Web Development
- Cyber Security
- Cloud Computing
- UI / UX


In [21]:
print(df.shape)
print(df.columns.tolist())
print(df["domain"].value_counts())

(200, 23)
['internship_id', 'posted_at', 'role', 'company', 'location', 'stipend', 'duration', 'deadline', 'skills', 'eligibility', 'perks', 'description', 'deadline_is_demo', 'domain', 'work_mode', 'internship_type', 'stipend_min', 'stipend_max', 'stipend_avg', 'duration_months', 'skills_list', 'start_type', 'start_date_raw']
domain
Software Development     66
AI / Machine Learning    32
Web Development          25
Data Science             23
Cloud Computing          12
UI / UX                   6
Cyber Security            6
Name: count, dtype: int64


In [22]:
# Step 15 - Calculate Domain Match Score

student_domain = "AI / Machine Learning"

# Direct domain matching
internship_df["domain_match_score"] = (    internship_df["domain"] == student_domain
).astype(int)

# Convert to 0-100
internship_df["domain_match_100"] = (
    internship_df["domain_match_score"] * 100
)

print(
    internship_df[
        ["role", "domain", "domain_match_100"]
    ].head(15)
)

                       role                 domain  domain_match_100
0      Full Stack Developer        Web Development                 0
1              UI/UX Design                UI / UX                 0
2        Frontend Developer        Web Development                 0
3        Frontend Developer        Web Development                 0
4       Flutter Development   Software Development                 0
5         Backend Developer        Web Development                 0
6            Data Analytics           Data Science                 0
7        Frontend Developer        Web Development                 0
8   Artificial Intelligence  AI / Machine Learning               100
9       Android Development   Software Development                 0
10     Software Development   Software Development                 0
11           Data Analytics           Data Science                 0
12         Embedded Systems   Software Development                 0
13            Cybersecurity       

In [23]:
# Step 15 - Handle "Any Domain" preference

if student_domain == "Any Domain":
    internship_df["domain_match_100"] = 100
else:
    internship_df["domain_match_100"] = (
        internship_df["domain"] == student_domain
    ).astype(int) * 100

print(
    internship_df[
        ["role", "domain", "domain_match_100"]
    ].head(15)
)

                       role                 domain  domain_match_100
0      Full Stack Developer        Web Development                 0
1              UI/UX Design                UI / UX                 0
2        Frontend Developer        Web Development                 0
3        Frontend Developer        Web Development                 0
4       Flutter Development   Software Development                 0
5         Backend Developer        Web Development                 0
6            Data Analytics           Data Science                 0
7        Frontend Developer        Web Development                 0
8   Artificial Intelligence  AI / Machine Learning               100
9       Android Development   Software Development                 0
10     Software Development   Software Development                 0
11           Data Analytics           Data Science                 0
12         Embedded Systems   Software Development                 0
13            Cybersecurity       

In [24]:
# Step 16 - Inspect internship work mode information

internship_df["location"] = internship_df["location"].replace({
    "Bangalore": "Bengaluru"
})

print("Unique locations:")
print(internship_df["location"].unique())

print("\nLocation distribution:")
print(internship_df["location"].value_counts())



Unique locations:
<StringArray>
[       'Remote',        'Mumbai',         'Noida',       'Chennai',
     'Bengaluru',     'Hyderabad',          'Pune',         'Delhi',
       'Gurgaon',         'Thane', 'Greater Noida',      'Dehradun']
Length: 12, dtype: str

Location distribution:
location
Remote           41
Chennai          28
Mumbai           23
Bengaluru        22
Delhi            21
Noida            19
Hyderabad        19
Pune             18
Thane             3
Greater Noida     3
Gurgaon           2
Dehradun          1
Name: count, dtype: int64


In [25]:
# Step 16 - Verify Work Mode

print(internship_df["work_mode"].value_counts(dropna=False))

work_mode
On-site    106
Hybrid      53
Remote      41
Name: count, dtype: int64


In [26]:
# Step 16 - Calculate Work Mode Match Score

student_work_mode = "Remote"

def calculate_work_mode_score(internship_mode, student_mode):
    # Any means no work-mode restriction
    if student_mode == "Any":
        return 100

    # Exact match
    if internship_mode == student_mode:
        return 100

    # Hybrid can be a partial match for Remote or On-site
    if student_mode in ["Remote", "On-site"] and internship_mode == "Hybrid":
        return 50

    # Otherwise no match
    return 0


internship_df["work_mode_match_100"] = internship_df["work_mode"].apply(
    lambda mode: calculate_work_mode_score(
        mode,
        student_work_mode
    )
)

print(
    internship_df[
        ["role", "location", "work_mode", "work_mode_match_100"]
    ].head(20)
)

                       role location work_mode  work_mode_match_100
0      Full Stack Developer   Remote    Remote                  100
1              UI/UX Design   Remote    Remote                  100
2        Frontend Developer   Remote    Remote                  100
3        Frontend Developer   Mumbai    Hybrid                   50
4       Flutter Development   Remote    Remote                  100
5         Backend Developer   Remote    Remote                  100
6            Data Analytics   Remote    Remote                  100
7        Frontend Developer   Remote    Remote                  100
8   Artificial Intelligence   Mumbai   On-site                    0
9       Android Development   Remote    Remote                  100
10     Software Development    Noida   On-site                    0
11           Data Analytics  Chennai    Hybrid                   50
12         Embedded Systems  Chennai   On-site                    0
13            Cybersecurity  Chennai    Hybrid  

In [27]:
# Step 17A - Setup Nominatim Geocoding

import requests
import time

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"

HEADERS = {
    "User-Agent": "InternSetu/1.0 (internship recommendation project)"
}

def geocode_city(city):
    if pd.isna(city):
        return None, None

    city = str(city).strip()

    if city.lower() == "remote":
        return None, None

    params = {
        "q": f"{city}, India",
        "format": "json",
        "limit": 1
    }

    response = requests.get(
        NOMINATIM_URL,
        params=params,
        headers=HEADERS,
        timeout=10
    )

    response.raise_for_status()

    results = response.json()

    if not results:
        return None, None

    return float(results[0]["lat"]), float(results[0]["lon"])

In [28]:
'''# Step 17B - Geocode internship locations and store coordinates

location_coordinates = {}

for location in unique_locations:

    if str(location).strip().lower() == "remote":
        location_coordinates[location] = {
            "latitude": np.nan,
            "longitude": np.nan
        }
        continue

    lat, lon = geocode_city(location)

    location_coordinates[location] = {
        "latitude": lat,
        "longitude": lon
    }

    time.sleep(1)

# Add coordinates to internship dataset
internship_df["latitude"] = internship_df["location"].map(
    lambda x: location_coordinates.get(x, {}).get("latitude", np.nan)
)

internship_df["longitude"] = internship_df["location"].map(
    lambda x: location_coordinates.get(x, {}).get("longitude", np.nan)
)

print(
    internship_df[
        ["role", "location", "latitude", "longitude"]
    ].head(10)
)'''

'# Step 17B - Geocode internship locations and store coordinates\n\nlocation_coordinates = {}\n\nfor location in unique_locations:\n\n    if str(location).strip().lower() == "remote":\n        location_coordinates[location] = {\n            "latitude": np.nan,\n            "longitude": np.nan\n        }\n        continue\n\n    lat, lon = geocode_city(location)\n\n    location_coordinates[location] = {\n        "latitude": lat,\n        "longitude": lon\n    }\n\n    time.sleep(1)\n\n# Add coordinates to internship dataset\ninternship_df["latitude"] = internship_df["location"].map(\n    lambda x: location_coordinates.get(x, {}).get("latitude", np.nan)\n)\n\ninternship_df["longitude"] = internship_df["location"].map(\n    lambda x: location_coordinates.get(x, {}).get("longitude", np.nan)\n)\n\nprint(\n    internship_df[\n        ["role", "location", "latitude", "longitude"]\n    ].head(10)\n)'

In [29]:
# Step 17B - Geocode internship locations and store coordinates

import time

# Get all unique internship locations
unique_locations = sorted(
    internship_df["location"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Locations to geocode:")
print(unique_locations)

# Dictionary to store coordinates
location_coordinates = {}

for location in unique_locations:

    # Remote has no physical location
    if location.lower() in ["remote", "work from home"]:
        location_coordinates[location] = {
            "latitude": np.nan,
            "longitude": np.nan
        }
        continue

    # Geocode city
    lat, lon = geocode_city(location)

    location_coordinates[location] = {
        "latitude": lat,
        "longitude": lon
    }

    # Respect Nominatim public usage limit
    time.sleep(1)

# Add latitude and longitude to internship dataframe
internship_df["latitude"] = internship_df["location"].map(
    lambda x: location_coordinates.get(
        x, {}
    ).get("latitude", np.nan)
)

internship_df["longitude"] = internship_df["location"].map(
    lambda x: location_coordinates.get(
        x, {}
    ).get("longitude", np.nan)
)

print("\nLocation Coordinates:")
print(location_coordinates)

print("\nSample:")
print(
    internship_df[
        ["role", "location", "latitude", "longitude"]
    ].head(10)
)

Locations to geocode:
['Bengaluru', 'Chennai', 'Dehradun', 'Delhi', 'Greater Noida', 'Gurgaon', 'Hyderabad', 'Mumbai', 'Noida', 'Pune', 'Remote', 'Thane']

Location Coordinates:
{'Bengaluru': {'latitude': 12.9767936, 'longitude': 77.590082}, 'Chennai': {'latitude': 13.0836939, 'longitude': 80.270186}, 'Dehradun': {'latitude': 30.3255646, 'longitude': 78.0436813}, 'Delhi': {'latitude': 28.6328027, 'longitude': 77.2197713}, 'Greater Noida': {'latitude': 28.4670734, 'longitude': 77.5137649}, 'Gurgaon': {'latitude': 28.4646148, 'longitude': 77.0299194}, 'Hyderabad': {'latitude': 17.360589, 'longitude': 78.4740613}, 'Mumbai': {'latitude': 19.054999, 'longitude': 72.8692035}, 'Noida': {'latitude': 28.5706333, 'longitude': 77.3272147}, 'Pune': {'latitude': 18.5213738, 'longitude': 73.8545071}, 'Remote': {'latitude': nan, 'longitude': nan}, 'Thane': {'latitude': 19.3653598, 'longitude': 73.3685437}}

Sample:
                      role location   latitude  longitude
0     Full Stack Developer  

In [30]:
# Step 17C - Geocode student location

student_city = "Mumbai"

student_lat, student_lon = geocode_city(student_city)

print("Student City:", student_city)
print("Latitude:", student_lat)
print("Longitude:", student_lon)

Student City: Mumbai
Latitude: 19.054999
Longitude: 72.8692035


In [31]:
# Step 17D - Calculate distance between student and internship locations

from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        return np.nan

    R = 6371  # Earth radius in kilometers

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


# Calculate distance for every internship
internship_df["distance_km"] = internship_df.apply(
    lambda row: haversine_distance(
        student_lat,
        student_lon,
        row["latitude"],
        row["longitude"]
    ),
    axis=1
)

# Display results
print(
    internship_df[
        ["role", "location", "distance_km"]
    ].head(20)
)

                       role location  distance_km
0      Full Stack Developer   Remote          NaN
1              UI/UX Design   Remote          NaN
2        Frontend Developer   Remote          NaN
3        Frontend Developer   Mumbai     0.000000
4       Flutter Development   Remote          NaN
5         Backend Developer   Remote          NaN
6            Data Analytics   Remote          NaN
7        Frontend Developer   Remote          NaN
8   Artificial Intelligence   Mumbai     0.000000
9       Android Development   Remote          NaN
10     Software Development    Noida  1150.850959
11           Data Analytics  Chennai  1032.208010
12         Embedded Systems  Chennai  1032.208010
13            Cybersecurity  Chennai  1032.208010
14         Machine Learning   Remote          NaN
15            Generative AI  Chennai  1032.208010
16         Software Testing   Remote          NaN
17     Software Development   Remote          NaN
18                 Robotics   Remote          NaN


In [32]:
# Step 17E - Convert distance into Location Match Score

def distance_to_score(distance):
    if pd.isna(distance):
        return np.nan

    # Distance cannot be negative
    distance = max(distance, 0)

    # Exponential decay:
    # Nearby locations get higher scores,
    # while distant locations gradually get lower scores.
    score = 100 * np.exp(-distance / 500)

    return round(score, 2)


internship_df["location_score_100"] = internship_df[
    "distance_km"
].apply(distance_to_score)

print(
    internship_df[
        ["role", "location", "distance_km", "location_score_100"]
    ]
    .sort_values("distance_km", na_position="last")
    .head(20)
)

                        role location  distance_km  location_score_100
3         Frontend Developer   Mumbai          0.0               100.0
8    Artificial Intelligence   Mumbai          0.0               100.0
23      Database Engineering   Mumbai          0.0               100.0
42          Data Engineering   Mumbai          0.0               100.0
50          Embedded Systems   Mumbai          0.0               100.0
82           IoT Development   Mumbai          0.0               100.0
66              Data Science   Mumbai          0.0               100.0
58        System Programming   Mumbai          0.0               100.0
74                    DevOps   Mumbai          0.0               100.0
138            Cybersecurity   Mumbai          0.0               100.0
154               CAD Design   Mumbai          0.0               100.0
122             UI/UX Design   Mumbai          0.0               100.0
114                 Robotics   Mumbai          0.0               100.0
106   

In [33]:
# Step 18 - Calculate Stipend Match Score

student_min_stipend = 10000


def calculate_stipend_score(stipend_min, student_min_stipend):
    # Missing stipend information
    if pd.isna(stipend_min):
        return np.nan

    # Internship meets or exceeds student's minimum
    if stipend_min >= student_min_stipend:
        return 100

    # Partial score when stipend is below the student's preference
    score = (stipend_min / student_min_stipend) * 100

    return round(max(0, min(score, 100)), 2)


internship_df["stipend_score_100"] = internship_df[
    "stipend_min"
].apply(
    lambda x: calculate_stipend_score(
        x,
        student_min_stipend
    )
)

print(
    internship_df[
        [
            "role",
            "stipend_min",
            "stipend_max",
            "stipend_score_100"
        ]
    ].head(20)
)

                       role  stipend_min  stipend_max  stipend_score_100
0      Full Stack Developer      10000.0      10000.0             100.00
1              UI/UX Design       5000.0      10000.0              50.00
2        Frontend Developer       1000.0       1000.0              10.00
3        Frontend Developer      10000.0      15000.0             100.00
4       Flutter Development       1001.0       5000.0              10.01
5         Backend Developer      12000.0      12000.0             100.00
6            Data Analytics      15000.0      25000.0             100.00
7        Frontend Developer      10000.0      10000.0             100.00
8   Artificial Intelligence       8000.0      12000.0              80.00
9       Android Development      20000.0      50000.0             100.00
10     Software Development      15000.0      15000.0             100.00
11           Data Analytics      20000.0      30000.0             100.00
12         Embedded Systems      20000.0      35000

In [34]:
# Step 19 - Calculate Duration Match Score

student_preferred_duration = 3  # months


def calculate_duration_score(
    internship_duration,
    preferred_duration
):
    # Missing duration information
    if pd.isna(internship_duration):
        return np.nan

    # Exact match
    if internship_duration == preferred_duration:
        return 100

    # Difference from student's preference
    difference = abs(
        internship_duration - preferred_duration
    )

    # Score decreases as duration difference increases
    score = 100 / (1 + difference)

    return round(score, 2)


internship_df["duration_score_100"] = internship_df[
    "duration_months"
].apply(
    lambda x: calculate_duration_score(
        x,
        student_preferred_duration
    )
)

print(
    internship_df[
        [
            "role",
            "duration_months",
            "duration_score_100"
        ]
    ].head(20)
)

                       role  duration_months  duration_score_100
0      Full Stack Developer             3.00              100.00
1              UI/UX Design             2.00               50.00
2        Frontend Developer             6.00               25.00
3        Frontend Developer             6.00               25.00
4       Flutter Development             1.00               33.33
5         Backend Developer             6.00               25.00
6            Data Analytics             3.00              100.00
7        Frontend Developer             3.00              100.00
8   Artificial Intelligence             6.00               25.00
9       Android Development             6.00               25.00
10     Software Development             3.00              100.00
11           Data Analytics             6.00               25.00
12         Embedded Systems             6.00               25.00
13            Cybersecurity             6.00               25.00
14         Machine Learni

In [35]:
internship_df.columns

Index(['internship_id', 'role', 'company', 'location', 'skills', 'skills_list',
       'eligibility', 'stipend_min', 'stipend_max', 'stipend_avg',
       'duration_months', 'start_type', 'deadline', 'description', 'domain',
       'work_mode', 'internship_type', 'normalized_skills',
       'skill_match_score', 'skill_match_score_100', 'role_similarity',
       'role_similarity_100', 'domain_match_score', 'domain_match_100',
       'work_mode_match_100', 'latitude', 'longitude', 'distance_km',
       'location_score_100', 'stipend_score_100', 'duration_score_100'],
      dtype='str')

In [36]:
# Step 20 - Calculate Internship Type Match Score

student_internship_type = "Non Technical"


def calculate_internship_type_score(
    internship_type,
    student_type
):
    # Missing internship type
    if pd.isna(internship_type):
        return np.nan

    # Any = no preference
    if student_type == "Any":
        return 100

    # Exact type match
    if internship_type == student_type:
        return 100

    # Different type = low preference
    return 0


internship_df["internship_type_score_100"] = internship_df[
    "internship_type"
].apply(
    lambda x: calculate_internship_type_score(
        x,
        student_internship_type
    )
)

print(
    internship_df[
        [
            "role",
            "internship_type",
            "internship_type_score_100"
        ]
    ].head(20)
)

                       role internship_type  internship_type_score_100
0      Full Stack Developer       Technical                          0
1              UI/UX Design       Technical                          0
2        Frontend Developer       Technical                          0
3        Frontend Developer       Technical                          0
4       Flutter Development       Technical                          0
5         Backend Developer       Technical                          0
6            Data Analytics       Technical                          0
7        Frontend Developer       Technical                          0
8   Artificial Intelligence       Technical                          0
9       Android Development       Technical                          0
10     Software Development       Technical                          0
11           Data Analytics       Technical                          0
12         Embedded Systems       Technical                          0
13    

In [37]:
# Step 16 - Verify Work Mode

print(internship_df["work_mode"].value_counts(dropna=False))

work_mode
On-site    106
Hybrid      53
Remote      41
Name: count, dtype: int64


In [38]:
# Step 21 - Calculate Semantic Interest Match Score

# Optional student interest / work preference
student_interest_text = (
    "I want to work on web development, machine learning "
     "and data analysis."
)

# Generate embedding for student interest
student_interest_embedding = model.encode(
    [student_interest_text],
    normalize_embeddings=True
)

# Prepare internship descriptions
internship_descriptions = (
    internship_df["description"]
    .fillna("")
    .astype(str)
    .tolist()
)

# Generate embeddings for internship descriptions
internship_description_embeddings = model.encode(
    internship_descriptions,
    normalize_embeddings=True,
    show_progress_bar=True
)

# Calculate semantic similarity
interest_similarity = cosine_similarity(
    student_interest_embedding,
    internship_description_embeddings
)[0]

# Store raw semantic similarity
internship_df["interest_similarity"] = interest_similarity

# Convert to 0-100 scale
internship_df["interest_score_100"] = (
    internship_df["interest_similarity"] * 100
)

print(
    internship_df[
        [
            "role",
            "interest_similarity",
            "interest_score_100"
        ]
    ]
    .sort_values(
        "interest_similarity",
        ascending=False
    )
    .head(10)
)

Batches: 100%|██████████| 7/7 [00:00<00:00, 12.05it/s]

                       role  interest_similarity  interest_score_100
95     Full Stack Developer             0.531854           53.185398
64     Full Stack Developer             0.526253           52.625336
2        Frontend Developer             0.522841           52.284115
93       Frontend Developer             0.522072           52.207172
85   Blockchain Development             0.514989           51.498924
62       Frontend Developer             0.513872           51.387215
126    Full Stack Developer             0.513486           51.348614
147  Blockchain Development             0.512739           51.273853
54   Blockchain Development             0.500986           50.098621
63        Backend Developer             0.500907           50.090702


In [39]:
# Step 22A - Check available feature columns

print(internship_df.columns.tolist())

['internship_id', 'role', 'company', 'location', 'skills', 'skills_list', 'eligibility', 'stipend_min', 'stipend_max', 'stipend_avg', 'duration_months', 'start_type', 'deadline', 'description', 'domain', 'work_mode', 'internship_type', 'normalized_skills', 'skill_match_score', 'skill_match_score_100', 'role_similarity', 'role_similarity_100', 'domain_match_score', 'domain_match_100', 'work_mode_match_100', 'latitude', 'longitude', 'distance_km', 'location_score_100', 'stipend_score_100', 'duration_score_100', 'internship_type_score_100', 'interest_similarity', 'interest_score_100']


In [40]:
# Step 22A - Check all generated score columns

score_columns = [
    "skill_match_score_100",
    "role_similarity_100",
    "domain_match_100",
    "work_mode_match_100",
    "location_score_100",
    "stipend_score_100",
    "duration_score_100",
    "internship_type_score_100",
    "interest_score_100"
]

for col in score_columns:
    print(f"{col:35} → {col in internship_df.columns}")

skill_match_score_100               → True
role_similarity_100                 → True
domain_match_100                    → True
work_mode_match_100                 → True
location_score_100                  → True
stipend_score_100                   → True
duration_score_100                  → True
internship_type_score_100           → True
interest_score_100                  → True


In [41]:
# Step 22 - Create Final Feature Matrix

feature_columns = [
    "skill_match_score_100",
    "role_similarity_100",
    "domain_match_100",
    "work_mode_match_100",
    "location_score_100",
    "stipend_score_100",
    "duration_score_100",
    "internship_type_score_100",
    "interest_score_100"
]

# Verify that all required features exist
missing_features = [
    col for col in feature_columns
    if col not in internship_df.columns
]

print("Missing feature columns:", missing_features)

# Create final feature matrix
feature_matrix = internship_df[feature_columns].copy()

print("\nFeature Matrix Shape:")
print(feature_matrix.shape)

print("\nFeature Columns:")
print(feature_matrix.columns.tolist())

print("\nSample:")
display(feature_matrix.head(10))

Missing feature columns: []

Feature Matrix Shape:
(200, 9)

Feature Columns:
['skill_match_score_100', 'role_similarity_100', 'domain_match_100', 'work_mode_match_100', 'location_score_100', 'stipend_score_100', 'duration_score_100', 'internship_type_score_100', 'interest_score_100']

Sample:


,skill_match_score_100,role_similarity_100,domain_match_100,work_mode_match_100,location_score_100,stipend_score_100,duration_score_100,internship_type_score_100,interest_score_100
0,31.123963,40.336235,0,100,NaN,100.00,100.00,0,38.374367
1,25.346366,14.217605,0,100,NaN,50.00,50.00,0,25.015242
2,31.123963,33.503769,0,100,NaN,10.00,25.00,0,52.284115
3,31.123963,33.503769,0,50,100.0,100.00,25.00,0,45.058723
4,25.420347,13.402804,0,100,NaN,10.01,33.33,0,19.067673
5,78.710914,32.770367,0,100,NaN,100.00,25.00,0,41.227238
6,80.144525,39.404720,0,100,NaN,100.00,100.00,0,40.033466
7,31.123963,33.503769,0,100,NaN,100.00,100.00,0,31.305916
8,66.289055,53.425938,100,0,100.0,80.00,25.00,0,39.619831
9,37.726125,23.437157,0,100,NaN,100.00,25.00,0,23.614635


In [42]:
# Step 23 - Final Feature Validation

feature_columns = [
    "skill_match_score_100",
    "role_similarity_100",
    "domain_match_100",
    "work_mode_match_100",
    "location_score_100",
    "stipend_score_100",
    "duration_score_100",
    "internship_type_score_100",
    "interest_score_100"
]

# Check missing values
print("Missing values in features:")
print(internship_df[feature_columns].isna().sum())

print("\nPercentage of missing values:")
print(
    (internship_df[feature_columns].isna().mean() * 100)
    .round(2)
)

Missing values in features:
skill_match_score_100         0
role_similarity_100           0
domain_match_100              0
work_mode_match_100           0
location_score_100           41
stipend_score_100             1
duration_score_100            0
internship_type_score_100     0
interest_score_100            0
dtype: int64

Percentage of missing values:
skill_match_score_100         0.0
role_similarity_100           0.0
domain_match_100              0.0
work_mode_match_100           0.0
location_score_100           20.5
stipend_score_100             0.5
duration_score_100            0.0
internship_type_score_100     0.0
interest_score_100            0.0
dtype: float64


In [43]:
# Step 23A - Identify missing feature values

print("Location score missing:")
print(
    internship_df.loc[
        internship_df["location_score_100"].isna(),
        ["role", "location", "work_mode"]
    ].head(20)
)

print("\nStipend score missing:")
print(
    internship_df.loc[
        internship_df["stipend_score_100"].isna(),
        ["role", "stipend_min", "stipend_max"]
    ]
)

Location score missing:
                        role location work_mode
0       Full Stack Developer   Remote    Remote
1               UI/UX Design   Remote    Remote
2         Frontend Developer   Remote    Remote
4        Flutter Development   Remote    Remote
5          Backend Developer   Remote    Remote
6             Data Analytics   Remote    Remote
7         Frontend Developer   Remote    Remote
9        Android Development   Remote    Remote
14          Machine Learning   Remote    Remote
16          Software Testing   Remote    Remote
17      Software Development   Remote    Remote
18                  Robotics   Remote    Remote
20              Data Science   Remote    Remote
22  React Native Development   Remote    Remote
25           iOS Development   Remote    Remote
28          Data Engineering   Remote    Remote
29           Cloud Computing   Remote    Remote
30    Blockchain Development   Remote    Remote
34          Software Testing   Remote    Remote
39              

In [44]:
# Step 1 - Eligibility Filtering

student_education = "Undergraduate"


def is_eligible(internship_eligibility, student_education):
    if pd.isna(internship_eligibility):
        return True

    eligibility = str(internship_eligibility).strip().lower()
    education = str(student_education).strip().lower()

    # Both undergraduate and postgraduate accepted
    if "undergraduate" in eligibility and "postgraduate" in eligibility:
        return True

    # Exact education match
    if education in eligibility:
        return True

    return False


internship_df["eligible"] = internship_df["eligibility"].apply(
    lambda x: is_eligible(
        x,
        student_education
    )
)

eligible_df = internship_df[
    internship_df["eligible"]
].copy()

print("Total internships:", len(internship_df))
print("Eligible internships:", len(eligible_df))

print(
    eligible_df[
        ["role", "eligibility", "eligible"]
    ].head(20)
)

Total internships: 200
Eligible internships: 136
                        role                   eligibility  eligible
1               UI/UX Design  Postgraduate & Undergraduate      True
2         Frontend Developer  Postgraduate & Undergraduate      True
4        Flutter Development                 Undergraduate      True
5          Backend Developer  Postgraduate & Undergraduate      True
7         Frontend Developer                 Undergraduate      True
8    Artificial Intelligence  Postgraduate & Undergraduate      True
10      Software Development                 Undergraduate      True
11            Data Analytics                 Undergraduate      True
12          Embedded Systems                 Undergraduate      True
15             Generative AI                 Undergraduate      True
16          Software Testing                 Undergraduate      True
19          Game Development  Postgraduate & Undergraduate      True
20              Data Science  Postgraduate & Undergrad

In [45]:
# Step 2 - Student Explicit Preferences

student_preferences = {
    "internship_type": "Technical",
    "work_mode": "Remote",
    "preferred_domain": "AI / Machine Learning",
    "location_preference": "Same City",
    "location": "Mumbai",
    "minimum_stipend": 10000,
    "preferred_duration": 3
}

print("Student Preferences:")
for key, value in student_preferences.items():
    print(f"{key}: {value}")

Student Preferences:
internship_type: Technical
work_mode: Remote
preferred_domain: AI / Machine Learning
location_preference: Same City
location: Mumbai
minimum_stipend: 10000
preferred_duration: 3


In [46]:
# Step 3 - Explicit Preference Filtering

primary_conditions = (
    (eligible_df["internship_type"] == student_preferences["internship_type"]) &
    (eligible_df["work_mode"] == student_preferences["work_mode"]) &
    (eligible_df["domain"] == student_preferences["preferred_domain"])
)

# Location preference: Same City
if student_preferences["location_preference"] == "Same City":
    primary_conditions &= (
        eligible_df["location"].str.lower()
        == student_preferences["location"].lower()
    )

# Minimum stipend preference
primary_conditions &= (
    eligible_df["stipend_min"].fillna(0)
    >= student_preferences["minimum_stipend"]
)

# Preferred duration
primary_conditions &= (
    eligible_df["duration_months"]
    <= student_preferences["preferred_duration"]
)

# Primary candidates
primary_candidates = eligible_df[
    primary_conditions
].copy()

print("Eligible internships:", len(eligible_df))
print("Primary candidates:", len(primary_candidates))

print("\nPrimary candidate sample:")
display(
    primary_candidates[
        [
            "role",
            "internship_type",
            "work_mode",
            "domain",
            "location",
            "stipend_min",
            "duration_months"
        ]
    ].head(20)
)

Eligible internships: 136
Primary candidates: 0

Primary candidate sample:


,role,internship_type,work_mode,domain,location,stipend_min,duration_months


In [47]:
# Step 3A - Verify preference score columns

preference_score_columns = [
    "internship_type_score_100",
    "work_mode_match_100",
    "domain_match_100",
    "location_score_100",
    "stipend_score_100",
    "duration_score_100"
]

for col in preference_score_columns:
    print(col, "→", col in internship_df.columns)

internship_type_score_100 → True
work_mode_match_100 → True
domain_match_100 → True
location_score_100 → True
stipend_score_100 → True
duration_score_100 → True


In [48]:
# Step 3 - Calculate Overall Preference Score

preference_score_columns = [
    "internship_type_score_100",
    "work_mode_match_100",
    "domain_match_100",
    "location_score_100",
    "stipend_score_100",
    "duration_score_100"
]

# Select only eligible internships from internship_df
# This preserves all calculated feature columns.
preference_df = internship_df[
    internship_df["internship_id"].isin(
        eligible_df["internship_id"]
    )
].copy()

# Calculate overall preference score
# NaN values are ignored.
preference_df["preference_score_100"] = (
    preference_df[preference_score_columns]
    .mean(axis=1, skipna=True)
)

print("Eligible internships:", len(preference_df))

print("\nPreference Score Statistics:")
print(
    preference_df["preference_score_100"].describe()
)

print("\nTop Preference Matches:")

display(
    preference_df[
        [
            "role",
            "internship_type",
            "work_mode",
            "domain",
            "location",
            "preference_score_100"
        ]
    ]
    .sort_values(
        "preference_score_100",
        ascending=False
    )
    .head(20)
)

Eligible internships: 136

Preference Score Statistics:
count    136.000000
mean      36.621814
std       10.349834
min       10.828333
25%       28.522667
50%       37.500000
75%       43.446667
max       63.123333
Name: preference_score_100, dtype: float64

Top Preference Matches:


,role,internship_type,work_mode,domain,location,preference_score_100
101,NLP,Technical,Hybrid,AI / Machine Learning,Pune,63.123333
7,Frontend Developer,Technical,Remote,Web Development,Remote,60.000000
39,NLP,Technical,Remote,AI / Machine Learning,Remote,60.000000
71,Computer Vision,Technical,Remote,AI / Machine Learning,Remote,60.000000
188,Corporate Sales,Non-Technical,Remote,NaN,Remote,60.000000
72,Deep Learning,Technical,Hybrid,AI / Machine Learning,Bengaluru,58.086667
69,Generative AI,Technical,On-site,AI / Machine Learning,Pune,54.790000
165,Deep Learning,Technical,On-site,AI / Machine Learning,Pune,54.790000
162,Generative AI,Technical,On-site,AI / Machine Learning,Mumbai,54.166667
161,Artificial Intelligence,Technical,Hybrid,AI / Machine Learning,Noida,51.668333


In [49]:
# Step 4 - Preference-Aware Candidate Selection

# Check whether the student has provided any explicit preference
has_explicit_preferences = any([
    student_preferences.get("internship_type") not in [None, "", "Any"],
    student_preferences.get("work_mode") not in [None, "", "Any"],
    student_preferences.get("preferred_domain") not in [None, "", "Any Domain"],
    student_preferences.get("location_preference") not in [None, "", "Any City"],
    student_preferences.get("minimum_stipend") not in [None, ""],
    student_preferences.get("preferred_duration") not in [None, ""]
])

print("Has explicit preferences:", has_explicit_preferences)


if not has_explicit_preferences:
    # No explicit preferences → keep all eligible internships
    candidate_df = preference_df.copy()

else:
    # Preference-oriented candidates
    preference_top = (
        preference_df
        .sort_values(
            "preference_score_100",
            ascending=False
        )
        .head(80)
    )

    # Strong skill-match candidates are also retained
    # so that useful alternatives are not lost.
    skill_top = (
        preference_df
        .sort_values(
            "skill_match_score_100",
            ascending=False
        )
        .head(20)
    )

    # Combine both candidate groups
    candidate_df = pd.concat(
        [preference_top, skill_top]
    ).drop_duplicates(
        subset=["internship_id"]
    ).copy()

    # Keep candidate pool within the desired 60-100 range
    if len(candidate_df) > 100:
        candidate_df = (
            candidate_df
            .sort_values(
                "preference_score_100",
                ascending=False
            )
            .head(100)
            .copy()
        )


print("\nEligible internships:", len(preference_df))
print("Candidate internships:", len(candidate_df))

print("\nCandidate sample:")
display(
    candidate_df[
        [
            "role",
            "internship_type",
            "work_mode",
            "domain",
            "location",
            "skill_match_score_100",
            "preference_score_100"
        ]
    ].head(20)
)

Has explicit preferences: True

Eligible internships: 136
Candidate internships: 85

Candidate sample:


,role,internship_type,work_mode,domain,location,skill_match_score_100,preference_score_100
101,NLP,Technical,Hybrid,AI / Machine Learning,Pune,55.377060,63.123333
7,Frontend Developer,Technical,Remote,Web Development,Remote,31.123963,60.000000
39,NLP,Technical,Remote,AI / Machine Learning,Remote,57.198447,60.000000
71,Computer Vision,Technical,Remote,AI / Machine Learning,Remote,57.386082,60.000000
188,Corporate Sales,Non-Technical,Remote,NaN,Remote,35.065255,60.000000
72,Deep Learning,Technical,Hybrid,AI / Machine Learning,Bengaluru,57.386082,58.086667
69,Generative AI,Technical,On-site,AI / Machine Learning,Pune,55.377060,54.790000
165,Deep Learning,Technical,On-site,AI / Machine Learning,Pune,57.386082,54.790000
162,Generative AI,Technical,On-site,AI / Machine Learning,Mumbai,55.377060,54.166667
161,Artificial Intelligence,Technical,Hybrid,AI / Machine Learning,Noida,66.289055,51.668333


In [50]:
# Step 5 - Build Recommendation Scoring Logic

score_weights = {
    "skill_match_score_100": 0.25,
    "role_similarity_100": 0.15,
    "domain_match_100": 0.15,
    "interest_score_100": 0.15,
    "work_mode_match_100": 0.10,
    "location_score_100": 0.05,
    "stipend_score_100": 0.05,
    "duration_score_100": 0.05,
    "internship_type_score_100": 0.05
}

print("Total weight:", sum(score_weights.values()))

Total weight: 1.0


In [51]:
# Step 5 - Calculate weighted recommendation score

def calculate_recommendation_score(row, weights):
    weighted_sum = 0.0
    total_available_weight = 0.0

    for feature, weight in weights.items():

        value = row[feature]

        # Missing feature → do not penalize
        if pd.isna(value):
            continue

        weighted_sum += value * weight
        total_available_weight += weight

    # No usable features
    if total_available_weight == 0:
        return 0.0

    # Re-normalize available weights to 100
    score = weighted_sum / total_available_weight

    return round(score, 2)


candidate_df["recommendation_score_100"] = candidate_df.apply(
    lambda row: calculate_recommendation_score(
        row,
        score_weights
    ),
    axis=1
)

print(
    candidate_df[
        [
            "role",
            "skill_match_score_100",
            "domain_match_100",
            "interest_score_100",
            "recommendation_score_100"
        ]
    ].head(20)
)

                        role  skill_match_score_100  domain_match_100  \
101                      NLP              55.377060               100   
7         Frontend Developer              31.123963                 0   
39                       NLP              57.198447               100   
71           Computer Vision              57.386082               100   
188          Corporate Sales              35.065255                 0   
72             Deep Learning              57.386082               100   
69             Generative AI              55.377060               100   
165            Deep Learning              57.386082               100   
162            Generative AI              55.377060               100   
161  Artificial Intelligence              66.289055               100   
41             Deep Learning              57.386082               100   
8    Artificial Intelligence              66.289055               100   
20              Data Science              58.627802

In [52]:
# Step 6 - Handle Missing and Optional Preferences

def safe_preference_score(row, student_preferences):
    scores = []
    weights = []

    # Internship Type
    student_type = student_preferences.get("internship_type")

    if student_type not in [None, "", "Any"]:
        value = row.get("internship_type_score_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # Work Mode
    student_mode = student_preferences.get("work_mode")

    if student_mode not in [None, "", "Any"]:
        value = row.get("work_mode_match_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # Domain
    student_domain = student_preferences.get("preferred_domain")

    if student_domain not in [None, "", "Any Domain"]:
        value = row.get("domain_match_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # Location Preference
    location_pref = student_preferences.get("location_preference")

    if location_pref not in [None, "", "Any City"]:
        value = row.get("location_score_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # Minimum Stipend
    minimum_stipend = student_preferences.get("minimum_stipend")

    if minimum_stipend not in [None, ""]:
        value = row.get("stipend_score_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # Duration
    preferred_duration = student_preferences.get("preferred_duration")

    if preferred_duration not in [None, ""]:
        value = row.get("duration_score_100")
        if pd.notna(value):
            scores.append(value)
            weights.append(1.0)

    # No usable preferences
    if not scores:
        return 0.0

    # Average only available preference scores
    return round(
        np.average(scores, weights=weights),
        2
    )


# Test on current candidate set
candidate_df["safe_preference_score_100"] = candidate_df.apply(
    lambda row: safe_preference_score(
        row,
        student_preferences
    ),
    axis=1
)

print(
    candidate_df[
        [
            "role",
            "preference_score_100",
            "safe_preference_score_100"
        ]
    ].head(20)
)

                        role  preference_score_100  safe_preference_score_100
101                      NLP             63.123333                      63.12
7         Frontend Developer             60.000000                      60.00
39                       NLP             60.000000                      60.00
71           Computer Vision             60.000000                      60.00
188          Corporate Sales             60.000000                      60.00
72             Deep Learning             58.086667                      58.09
69             Generative AI             54.790000                      54.79
165            Deep Learning             54.790000                      54.79
162            Generative AI             54.166667                      54.17
161  Artificial Intelligence             51.668333                      51.67
41             Deep Learning             51.668333                      51.67
8    Artificial Intelligence             50.833333              

In [53]:
#                                               phase -5

In [54]:
# Step 1 - Calculate Final Recommendation Score

final_score_weights = {
    "skill_match_score_100": 0.25,
    "role_similarity_100": 0.15,
    "domain_match_100": 0.15,
    "interest_score_100": 0.15,
    "work_mode_match_100": 0.10,
    "location_score_100": 0.05,
    "stipend_score_100": 0.05,
    "duration_score_100": 0.05,
    "internship_type_score_100": 0.05
}

def calculate_final_score(row, weights):
    weighted_sum = 0.0
    total_available_weight = 0.0

    for feature, weight in weights.items():

        value = row.get(feature)

        # Missing feature should not unfairly reduce score
        if pd.isna(value):
            continue

        weighted_sum += value * weight
        total_available_weight += weight

    if total_available_weight == 0:
        return 0.0

    # Re-normalize available weights
    return round(
        weighted_sum / total_available_weight,
        2
    )


candidate_df["final_score_100"] = candidate_df.apply(
    lambda row: calculate_final_score(
        row,
        final_score_weights
    ),
    axis=1
)

print("Final scores calculated:",
      candidate_df["final_score_100"].notna().sum())

display(
    candidate_df[
        [
            "role",
            "skill_match_score_100",
            "role_similarity_100",
            "domain_match_100",
            "interest_score_100",
            "work_mode_match_100",
            "final_score_100"
        ]
    ].head(20)
)

Final scores calculated: 85


,role,skill_match_score_100,role_similarity_100,domain_match_100,interest_score_100,work_mode_match_100,final_score_100
101,NLP,55.377060,23.689556,100,27.836636,50,53.01
7,Frontend Developer,31.123963,33.503769,0,31.305916,100,39.48
39,NLP,57.198447,23.689556,100,27.370262,100,54.69
71,Computer Vision,57.386082,37.226902,100,31.248802,100,57.49
188,Corporate Sales,35.065255,12.027447,0,28.591125,100,36.69
72,Deep Learning,57.386082,51.230286,100,34.802402,50,57.18
69,Generative AI,55.377060,42.744568,100,24.344269,0,50.34
165,Deep Learning,57.386082,51.230286,100,34.959606,0,53.71
162,Generative AI,55.377060,42.744568,100,21.587345,0,49.74
161,Artificial Intelligence,66.289055,53.425938,100,38.074772,50,58.30


In [55]:
# Step 2 - Rank candidates by final recommendation score

ranked_df = (
    candidate_df
    .sort_values(
        "final_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

# Add rank
ranked_df["rank"] = ranked_df.index + 1

print("Total ranked candidates:", len(ranked_df))

display(
    ranked_df[
        [
            "rank",
            "role",
            "company",
            "final_score_100"
        ]
    ].head(20)
)

Total ranked candidates: 85


,rank,role,company,final_score_100
0,1,Machine Learning,CyberGrid,61.86
1,2,Machine Learning,TechBridge,60.29
2,3,Machine Learning,NextGen Technologies,58.47
3,4,Artificial Intelligence,Visionary AI,58.30
4,5,Computer Vision,CloudMatrix,57.49
5,6,Deep Learning,Nexora Labs,57.18
6,7,Artificial Intelligence,TechNova Solutions,56.86
7,8,Artificial Intelligence,Tax-O-Smart LLP,55.78
8,9,Artificial Intelligence,ByteWorks,55.25
9,10,Deep Learning,Visionary AI,55.13


In [56]:
# Step 3 - Select Top 5 Recommendations

top_5 = ranked_df.head(5).copy()

print("Top 5 Recommendations:")

display(
    top_5[
        [
            "rank",
            "role",
            "company",
            "location",
            "work_mode",
            "domain",
            "final_score_100"
        ]
    ]
)

Top 5 Recommendations:


,rank,role,company,location,work_mode,domain,final_score_100
0,1,Machine Learning,CyberGrid,Delhi,Hybrid,AI / Machine Learning,61.86
1,2,Machine Learning,TechBridge,Bengaluru,On-site,AI / Machine Learning,60.29
2,3,Machine Learning,NextGen Technologies,Noida,On-site,AI / Machine Learning,58.47
3,4,Artificial Intelligence,Visionary AI,Noida,Hybrid,AI / Machine Learning,58.30
4,5,Computer Vision,CloudMatrix,Remote,Remote,AI / Machine Learning,57.49


In [57]:
# Step 4 - Generate Recommendation Reasons

def generate_reasons(row):
    reasons = []

    if row["skill_match_score_100"] >= 60:
        reasons.append("Strong skill match")
    elif row["skill_match_score_100"] >= 40:
        reasons.append("Good skill match")

    if row["role_similarity_100"] >= 50:
        reasons.append("Strong role similarity")

    if row["domain_match_100"] >= 100:
        reasons.append("Preferred domain match")

    if row["work_mode_match_100"] >= 100:
        reasons.append("Preferred work mode")

    if pd.notna(row["location_score_100"]) and row["location_score_100"] >= 70:
        reasons.append("Good location match")

    if row["stipend_score_100"] >= 80:
        reasons.append("Meets stipend preference")

    if row["duration_score_100"] >= 80:
        reasons.append("Good duration match")

    if row["internship_type_score_100"] >= 100:
        reasons.append("Preferred internship type")

    if row["interest_score_100"] >= 50:
        reasons.append("Strong interest/description similarity")

    return reasons


top_5["recommendation_reasons"] = top_5.apply(
    generate_reasons,
    axis=1
)

display(
    top_5[
        [
            "rank",
            "role",
            "company",
            "final_score_100",
            "recommendation_reasons"
        ]
    ]
)

,rank,role,company,final_score_100,recommendation_reasons
0,1,Machine Learning,CyberGrid,61.86,"[Strong skill match, Strong role similarity, P..."
1,2,Machine Learning,TechBridge,60.29,"[Strong skill match, Strong role similarity, P..."
2,3,Machine Learning,NextGen Technologies,58.47,"[Strong skill match, Strong role similarity, P..."
3,4,Artificial Intelligence,Visionary AI,58.30,"[Strong skill match, Strong role similarity, P..."
4,5,Computer Vision,CloudMatrix,57.49,"[Good skill match, Preferred domain match, Pre..."


In [58]:
# Step 5 - Create Final Recommendation Output

recommendation_output = top_5[
    [
        "rank",
        "internship_id",
        "role",
        "company",
        "location",
        "work_mode",
        "domain",
        "stipend_min",
        "stipend_max",
        "duration_months",
        "final_score_100",
        "recommendation_reasons"
    ]
].copy()

print("Final Recommendation Output:")
display(recommendation_output)

Final Recommendation Output:


,rank,internship_id,role,company,location,work_mode,domain,stipend_min,stipend_max,duration_months,final_score_100,recommendation_reasons
0,1,2026INT0159,Machine Learning,CyberGrid,Delhi,Hybrid,AI / Machine Learning,12000.0,12000.0,6.0,61.86,"[Strong skill match, Strong role similarity, P..."
1,2,2026INT0097,Machine Learning,TechBridge,Bengaluru,On-site,AI / Machine Learning,8000.0,8000.0,3.0,60.29,"[Strong skill match, Strong role similarity, P..."
2,3,2026INT0066,Machine Learning,NextGen Technologies,Noida,On-site,AI / Machine Learning,10000.0,10000.0,4.0,58.47,"[Strong skill match, Strong role similarity, P..."
3,4,2026INT0162,Artificial Intelligence,Visionary AI,Noida,Hybrid,AI / Machine Learning,10000.0,10000.0,4.0,58.30,"[Strong skill match, Strong role similarity, P..."
4,5,2026INT0072,Computer Vision,CloudMatrix,Remote,Remote,AI / Machine Learning,5000.0,5000.0,2.0,57.49,"[Good skill match, Preferred domain match, Pre..."


In [59]:
# Step 6 - Validate Top 5 Recommendation Quality

print("Top 5 count:", len(top_5))
print("Unique internship IDs:", top_5["internship_id"].nunique())

print("\nTop 5 Scores:")
print(top_5["final_score_100"].tolist())

print("\nTop 5 Recommendations:")
display(
    top_5[
        [
            "rank",
            "internship_id",
            "role",
            "company",
            "location",
            "work_mode",
            "domain",
            "final_score_100",
            "recommendation_reasons"
        ]
    ]
)

# Basic validation
assert len(top_5) == 5
assert top_5["internship_id"].nunique() == 5
assert top_5["final_score_100"].is_monotonic_decreasing

print("\n✅ Top 5 recommendation quality checks passed.")

Top 5 count: 5
Unique internship IDs: 5

Top 5 Scores:
[61.86, 60.29, 58.47, 58.3, 57.49]

Top 5 Recommendations:


,rank,internship_id,role,company,location,work_mode,domain,final_score_100,recommendation_reasons
0,1,2026INT0159,Machine Learning,CyberGrid,Delhi,Hybrid,AI / Machine Learning,61.86,"[Strong skill match, Strong role similarity, P..."
1,2,2026INT0097,Machine Learning,TechBridge,Bengaluru,On-site,AI / Machine Learning,60.29,"[Strong skill match, Strong role similarity, P..."
2,3,2026INT0066,Machine Learning,NextGen Technologies,Noida,On-site,AI / Machine Learning,58.47,"[Strong skill match, Strong role similarity, P..."
3,4,2026INT0162,Artificial Intelligence,Visionary AI,Noida,Hybrid,AI / Machine Learning,58.30,"[Strong skill match, Strong role similarity, P..."
4,5,2026INT0072,Computer Vision,CloudMatrix,Remote,Remote,AI / Machine Learning,57.49,"[Good skill match, Preferred domain match, Pre..."



✅ Top 5 recommendation quality checks passed.


In [60]:
 #                                              phase 6

In [61]:
# Step 1 - Create Feedback Data Structure

feedback_data = []

def add_feedback(
    student_id,
    internship_id,
    feedback,
    domain,
    role
):
    feedback_entry = {
        "student_id": student_id,
        "internship_id": internship_id,
        "feedback": feedback,
        "domain": domain,
        "role": role
    }

    feedback_data.append(feedback_entry)


# Example feedback
add_feedback(
    student_id="student_001",
    internship_id="2026INT0159",
    feedback="like",
    domain="AI / Machine Learning",
    role="Machine Learning"
)

add_feedback(
    student_id="student_001",
    internship_id="2026INT0072",
    feedback="dislike",
    domain="AI / Machine Learning",
    role="Computer Vision"
)

print("Feedback records:")

for record in feedback_data:
    print(record)

Feedback records:
{'student_id': 'student_001', 'internship_id': '2026INT0159', 'feedback': 'like', 'domain': 'AI / Machine Learning', 'role': 'Machine Learning'}
{'student_id': 'student_001', 'internship_id': '2026INT0072', 'feedback': 'dislike', 'domain': 'AI / Machine Learning', 'role': 'Computer Vision'}


In [62]:
# Step 2 - Calculate Feedback Personalization Scores

feedback_df = pd.DataFrame(feedback_data)

# Convert feedback to numeric value
feedback_df["feedback_value"] = feedback_df["feedback"].map({
    "like": 1,
    "dislike": -1
})

print("Feedback Data:")
display(feedback_df)

# Student-level domain preference
domain_feedback = (
    feedback_df
    .groupby(["student_id", "domain"])["feedback_value"]
    .sum()
    .reset_index()
)

# Student-level role preference
role_feedback = (
    feedback_df
    .groupby(["student_id", "role"])["feedback_value"]
    .sum()
    .reset_index()
)

print("\nDomain Feedback:")
display(domain_feedback)

print("\nRole Feedback:")
display(role_feedback)

Feedback Data:


,student_id,internship_id,feedback,domain,role,feedback_value
0,student_001,2026INT0159,like,AI / Machine Learning,Machine Learning,1
1,student_001,2026INT0072,dislike,AI / Machine Learning,Computer Vision,-1



Domain Feedback:


,student_id,domain,feedback_value
0,student_001,AI / Machine Learning,0



Role Feedback:


,student_id,role,feedback_value
0,student_001,Computer Vision,-1
1,student_001,Machine Learning,1


In [63]:
'''# Step 3 - Create Normalized Personalization Scores

# Domain-level average feedback
domain_personalization = (
    feedback_df
    .groupby(["student_id", "domain"])["feedback_value"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "feedback_value": "domain_personalization"
        }
    )
)

# Role-level average feedback
role_personalization = (
    feedback_df
    .groupby(["student_id", "role"])["feedback_value"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "feedback_value": "role_personalization"
        }
    )
)

print("Domain Personalization:")
display(domain_personalization)

print("\nRole Personalization:")
display(role_personalization)'''

'# Step 3 - Create Normalized Personalization Scores\n\n# Domain-level average feedback\ndomain_personalization = (\n    feedback_df\n    .groupby(["student_id", "domain"])["feedback_value"]\n    .mean()\n    .reset_index()\n    .rename(\n        columns={\n            "feedback_value": "domain_personalization"\n        }\n    )\n)\n\n# Role-level average feedback\nrole_personalization = (\n    feedback_df\n    .groupby(["student_id", "role"])["feedback_value"]\n    .mean()\n    .reset_index()\n    .rename(\n        columns={\n            "feedback_value": "role_personalization"\n        }\n    )\n)\n\nprint("Domain Personalization:")\ndisplay(domain_personalization)\n\nprint("\nRole Personalization:")\ndisplay(role_personalization)'

In [64]:
# Step 3 - Calculate Cumulative Personalization Scores

# Domain-level cumulative feedback
domain_personalization = (
    feedback_df
    .groupby(["student_id", "domain"])["feedback_value"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "feedback_value": "domain_personalization"
        }
    )
)

# Role-level cumulative feedback
role_personalization = (
    feedback_df
    .groupby(["student_id", "role"])["feedback_value"]
    .sum()
    .reset_index()
    .rename(
        columns={
            "feedback_value": "role_personalization"
        }
    )
)

print("Domain Personalization:")
display(domain_personalization)

print("\nRole Personalization:")
display(role_personalization)

Domain Personalization:


,student_id,domain,domain_personalization
0,student_001,AI / Machine Learning,0



Role Personalization:


,student_id,role,role_personalization
0,student_001,Computer Vision,-1
1,student_001,Machine Learning,1


In [65]:
'''# Step 4 - Calculate Feedback Personalization Score

student_id = "student_001"

# Create lookup dictionaries for this student
domain_feedback_map = (
    domain_personalization[
        domain_personalization["student_id"] == student_id
    ]
    .set_index("domain")["domain_personalization"]
    .to_dict()
)

role_feedback_map = (
    role_personalization[
        role_personalization["student_id"] == student_id
    ]
    .set_index("role")["role_personalization"]
    .to_dict()
)


def calculate_feedback_score(row):
    domain_score = domain_feedback_map.get(
        row["domain"], 0
    )

    role_score = role_feedback_map.get(
        row["role"], 0
    )

    # Role feedback is more specific than domain feedback
    feedback_score = (
        0.6 * role_score +
        0.4 * domain_score
    )

    return feedback_score


candidate_df["feedback_personalization"] = candidate_df.apply(
    calculate_feedback_score,
    axis=1
)

print(
    candidate_df[
        [
            "role",
            "domain",
            "feedback_personalization"
        ]
    ].head(20)
)'''

'# Step 4 - Calculate Feedback Personalization Score\n\nstudent_id = "student_001"\n\n# Create lookup dictionaries for this student\ndomain_feedback_map = (\n    domain_personalization[\n        domain_personalization["student_id"] == student_id\n    ]\n    .set_index("domain")["domain_personalization"]\n    .to_dict()\n)\n\nrole_feedback_map = (\n    role_personalization[\n        role_personalization["student_id"] == student_id\n    ]\n    .set_index("role")["role_personalization"]\n    .to_dict()\n)\n\n\ndef calculate_feedback_score(row):\n    domain_score = domain_feedback_map.get(\n        row["domain"], 0\n    )\n\n    role_score = role_feedback_map.get(\n        row["role"], 0\n    )\n\n    # Role feedback is more specific than domain feedback\n    feedback_score = (\n        0.6 * role_score +\n        0.4 * domain_score\n    )\n\n    return feedback_score\n\n\ncandidate_df["feedback_personalization"] = candidate_df.apply(\n    calculate_feedback_score,\n    axis=1\n)\n\nprint(

In [66]:
# Step 4 - Calculate Student Personalization Signal
student_id = "student_001"
# Create role-level feedback lookup for the current student
role_feedback_map = (
    role_personalization[
        role_personalization["student_id"] == student_id
    ]
    .set_index("role")["role_personalization"]
    .to_dict()
)

# Create domain-level feedback lookup for the current student
domain_feedback_map = (
    domain_personalization[
        domain_personalization["student_id"] == student_id
    ]
    .set_index("domain")["domain_personalization"]
    .to_dict()
)


def calculate_feedback_score(row):
    # Get cumulative role feedback
    role_score = role_feedback_map.get(
        row["role"],
        0
    )

    # Get cumulative domain feedback
    domain_score = domain_feedback_map.get(
        row["domain"],
        0
    )

    # Role feedback gets more importance than domain feedback
    feedback_score = (
        0.6 * role_score +
        0.4 * domain_score
    )

    return feedback_score


# Apply personalization signal to candidate internships
candidate_df["feedback_personalization"] = candidate_df.apply(
    calculate_feedback_score,
    axis=1
)

print(
    candidate_df[
        [
            "role",
            "domain",
            "feedback_personalization"
        ]
    ].head(20)
)

                        role                 domain  feedback_personalization
101                      NLP  AI / Machine Learning                       0.0
7         Frontend Developer        Web Development                       0.0
39                       NLP  AI / Machine Learning                       0.0
71           Computer Vision  AI / Machine Learning                      -0.6
188          Corporate Sales                    NaN                       0.0
72             Deep Learning  AI / Machine Learning                       0.0
69             Generative AI  AI / Machine Learning                       0.0
165            Deep Learning  AI / Machine Learning                       0.0
162            Generative AI  AI / Machine Learning                       0.0
161  Artificial Intelligence  AI / Machine Learning                       0.0
41             Deep Learning  AI / Machine Learning                       0.0
8    Artificial Intelligence  AI / Machine Learning             

In [67]:
# Step 5 - Integrate Feedback Personalization

# Maximum influence that feedback can have on final score
feedback_influence = 10


def feedback_to_adjustment(feedback_score):
    """
    Convert cumulative feedback into a bounded score adjustment.

    Positive feedback  -> positive adjustment
    Negative feedback  -> negative adjustment
    More feedback      -> stronger influence
    """

    # No feedback
    if pd.isna(feedback_score):
        return 0.0

    # Bound the effect using tanh
    adjustment = np.tanh(feedback_score / 3) * feedback_influence

    return round(adjustment, 2)


# Convert cumulative personalization signal
candidate_df["feedback_adjustment"] = (
    candidate_df["feedback_personalization"]
    .apply(feedback_to_adjustment)
)

# Add personalization adjustment to the existing final score
candidate_df["personalized_score_100"] = (
    candidate_df["final_score_100"]
    + candidate_df["feedback_adjustment"]
)

# Keep score within 0-100
candidate_df["personalized_score_100"] = (
    candidate_df["personalized_score_100"]
    .clip(0, 100)
)

print(
    candidate_df[
        [
            "role",
            "final_score_100",
            "feedback_personalization",
            "feedback_adjustment",
            "personalized_score_100"
        ]
    ]
    .sort_values(
        "personalized_score_100",
        ascending=False
    )
    .head(20)
)

                        role  final_score_100  feedback_personalization  \
158         Machine Learning            61.86                       0.6   
96          Machine Learning            60.29                       0.6   
65          Machine Learning            58.47                       0.6   
161  Artificial Intelligence            58.30                       0.0   
72             Deep Learning            57.18                       0.0   
99   Artificial Intelligence            56.86                       0.0   
8    Artificial Intelligence            55.78                       0.0   
71           Computer Vision            57.49                      -0.6   
68   Artificial Intelligence            55.25                       0.0   
41             Deep Learning            55.13                       0.0   
39                       NLP            54.69                       0.0   
165            Deep Learning            53.71                       0.0   
101                      

In [68]:
# Step 6 - Personalized Ranking and Top 5

personalized_ranked_df = (
    candidate_df
    .sort_values(
        "personalized_score_100",
        ascending=False
    )
    .reset_index(drop=True)
)

# Assign new personalized rank
personalized_ranked_df["personalized_rank"] = (
    personalized_ranked_df.index + 1
)

print("Personalized Ranking:")

display(
    personalized_ranked_df[
        [
            "personalized_rank",
            "internship_id",
            "role",
            "company",
            "feedback_personalization",
            "final_score_100",
            "personalized_score_100"
        ]
    ].head(20)
)

Personalized Ranking:


,personalized_rank,internship_id,role,company,feedback_personalization,final_score_100,personalized_score_100
0,1,2026INT0159,Machine Learning,CyberGrid,0.6,61.86,63.83
1,2,2026INT0097,Machine Learning,TechBridge,0.6,60.29,62.26
2,3,2026INT0066,Machine Learning,NextGen Technologies,0.6,58.47,60.44
3,4,2026INT0162,Artificial Intelligence,Visionary AI,0.0,58.30,58.30
4,5,2026INT0073,Deep Learning,Nexora Labs,0.0,57.18,57.18
5,6,2026INT0100,Artificial Intelligence,TechNova Solutions,0.0,56.86,56.86
6,7,2025WSHP0827,Artificial Intelligence,Tax-O-Smart LLP,0.0,55.78,55.78
7,8,2026INT0072,Computer Vision,CloudMatrix,-0.6,57.49,55.52
8,9,2026INT0069,Artificial Intelligence,ByteWorks,0.0,55.25,55.25
9,10,2026INT0042,Deep Learning,Visionary AI,0.0,55.13,55.13


In [69]:
# Step 6 - Select Personalized Top 5

personalized_top_5 = personalized_ranked_df.head(5).copy()

print("Personalized Top 5:")

display(
    personalized_top_5[
        [
            "personalized_rank",
            "internship_id",
            "role",
            "company",
            "domain",
            "work_mode",
            "personalized_score_100"
        ]
    ]
)

Personalized Top 5:


,personalized_rank,internship_id,role,company,domain,work_mode,personalized_score_100
0,1,2026INT0159,Machine Learning,CyberGrid,AI / Machine Learning,Hybrid,63.83
1,2,2026INT0097,Machine Learning,TechBridge,AI / Machine Learning,On-site,62.26
2,3,2026INT0066,Machine Learning,NextGen Technologies,AI / Machine Learning,On-site,60.44
3,4,2026INT0162,Artificial Intelligence,Visionary AI,AI / Machine Learning,Hybrid,58.30
4,5,2026INT0073,Deep Learning,Nexora Labs,AI / Machine Learning,Hybrid,57.18


In [70]:
# Step 7 - Validate Feedback Impact on Recommendations

# Compare old score with personalized score
feedback_impact_df = candidate_df[
    [
        "internship_id",
        "role",
        "company",
        "final_score_100",
        "feedback_personalization",
        "feedback_adjustment",
        "personalized_score_100"
    ]
].copy()

# Calculate change caused by feedback
feedback_impact_df["score_change"] = (
    feedback_impact_df["personalized_score_100"]
    - feedback_impact_df["final_score_100"]
)

# Show internships affected by feedback
affected_df = (
    feedback_impact_df[
        feedback_impact_df["feedback_personalization"] != 0
    ]
    .sort_values(
        "score_change",
        ascending=False
    )
)

print("Internships affected by feedback:")
display(affected_df)

Internships affected by feedback:


,internship_id,role,company,final_score_100,feedback_personalization,feedback_adjustment,personalized_score_100,score_change
96,2026INT0097,Machine Learning,TechBridge,60.29,0.6,1.97,62.26,1.97
158,2026INT0159,Machine Learning,CyberGrid,61.86,0.6,1.97,63.83,1.97
65,2026INT0066,Machine Learning,NextGen Technologies,58.47,0.6,1.97,60.44,1.97
71,2026INT0072,Computer Vision,CloudMatrix,57.49,-0.6,-1.97,55.52,-1.97
164,2026INT0165,Computer Vision,QuantumSoft,49.57,-0.6,-1.97,47.60,-1.97
102,2026INT0103,Computer Vision,CodeCraft Labs,46.38,-0.6,-1.97,44.41,-1.97


In [71]:
# Step 7 - Validate Personalization Behavior

positive_feedback = feedback_impact_df[
    feedback_impact_df["feedback_personalization"] > 0
]

negative_feedback = feedback_impact_df[
    feedback_impact_df["feedback_personalization"] < 0
]

neutral_feedback = feedback_impact_df[
    feedback_impact_df["feedback_personalization"] == 0
]

print("Positive feedback internships:", len(positive_feedback))
print("Negative feedback internships:", len(negative_feedback))
print("Neutral internships:", len(neutral_feedback))

print("\nPositive feedback examples:")
display(
    positive_feedback[
        [
            "role",
            "final_score_100",
            "feedback_personalization",
            "personalized_score_100"
        ]
    ].head(10)
)

print("\nNegative feedback examples:")
display(
    negative_feedback[
        [
            "role",
            "final_score_100",
            "feedback_personalization",
            "personalized_score_100"
        ]
    ].head(10)
)

Positive feedback internships: 3
Negative feedback internships: 3
Neutral internships: 79

Positive feedback examples:


,role,final_score_100,feedback_personalization,personalized_score_100
96,Machine Learning,60.29,0.6,62.26
158,Machine Learning,61.86,0.6,63.83
65,Machine Learning,58.47,0.6,60.44



Negative feedback examples:


,role,final_score_100,feedback_personalization,personalized_score_100
71,Computer Vision,57.49,-0.6,55.52
164,Computer Vision,49.57,-0.6,47.60
102,Computer Vision,46.38,-0.6,44.41


In [72]:
#                                             phase -7

In [73]:
# Step 1 - Check Prisma database setup
# Step 6A - Check final recommendation objects

print("candidate_df shape:", candidate_df.shape)

print("\nTop 5:")
display(
    personalized_top_5[
        [
            "personalized_rank",
            "internship_id",
            "role",
            "company",
            "personalized_score_100"
        ]
    ]
)

print("\nAvailable notebook variables:")
print([
    "candidate_df",
    "ranked_df",
    "top_5",
    "personalized_ranked_df",
    "personalized_top_5"
])


candidate_df shape: (85, 42)

Top 5:


,personalized_rank,internship_id,role,company,personalized_score_100
0,1,2026INT0159,Machine Learning,CyberGrid,63.83
1,2,2026INT0097,Machine Learning,TechBridge,62.26
2,3,2026INT0066,Machine Learning,NextGen Technologies,60.44
3,4,2026INT0162,Artificial Intelligence,Visionary AI,58.30
4,5,2026INT0073,Deep Learning,Nexora Labs,57.18



Available notebook variables:
['candidate_df', 'ranked_df', 'top_5', 'personalized_ranked_df', 'personalized_top_5']
